# LeetCode 1286 — Iterator for Combination bằng Python

Link đề: https://leetcode.com/problems/iterator-for-combination/ · Độ khó: **Medium** · Chủ đề: String, Backtracking, Design, Iterator

Skeleton LeetCode (bài **thiết kế lớp**):

```python
class CombinationIterator:

    def __init__(self, characters: str, combinationLength: int):
        ...

    def next(self) -> str:
        ...

    def hasNext(self) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu **tổ hợp chập `k`** của một chuỗi và **thứ tự từ điển** (lexicographic) của chúng.
2. Thiết kế một lớp **iterator** với hai thao tác `next()` / `hasNext()` và biết viết **bộ chạy chuỗi lệnh kiểu LeetCode** để test.
3. Đi từ **sinh trước toàn bộ** (itertools / backtracking) → **bitmask** → **sinh tổ hợp kế tiếp tại chỗ (lazy)** chỉ tốn `O(k)` bộ nhớ.
4. Nắm thuật toán "**tổ hợp kế tiếp**": tìm vị trí phải nhất còn tăng được, tăng nó, đặt lại phần đuôi.
5. Tránh lỗi: `hasNext` lệch 1, bitmask sai thứ tự, generator bị "ăn mất" phần tử khi kiểm tra `hasNext`.

## 2. Đề bài & ví dụ

Thiết kế lớp `CombinationIterator`:

- `CombinationIterator(characters, combinationLength)`: khởi tạo với chuỗi `characters` gồm các chữ cái thường **phân biệt, đã sắp xếp tăng**, và số `combinationLength`.
- `next()`: trả về **tổ hợp kế tiếp** độ dài `combinationLength` theo **thứ tự từ điển**.
- `hasNext()`: trả `True` khi và chỉ khi còn tổ hợp kế tiếp.

**Ví dụ 1:**
```
Input : ["CombinationIterator", "next", "hasNext", "next", "hasNext", "next", "hasNext"]
        [["abc", 2], [], [], [], [], [], []]
Output: [null, "ab", true, "ac", true, "bc", false]
```

**Ràng buộc:**
- `1 <= combinationLength <= characters.length <= 15`
- Các ký tự trong `characters` phân biệt.
- Tối đa `10^4` lời gọi `next` và `hasNext`.
- Mọi lời gọi `next` đều hợp lệ (còn tổ hợp).

In [1]:
import random
import time
from itertools import combinations
from math import comb
from typing import List

EX_OPS = ["CombinationIterator", "next", "hasNext", "next", "hasNext", "next", "hasNext"]
EX_ARGS = [["abc", 2], [], [], [], [], [], []]
EX_OUT = [None, "ab", True, "ac", True, "bc", False]

print("Các tổ hợp chập 2 của 'abc' theo thứ tự từ điển:",
      ["".join(c) for c in combinations("abc", 2)])
for op, a, out in zip(EX_OPS, EX_ARGS, EX_OUT):
    print(f"{op:<20} {str(a):<12} -> {out}")

Các tổ hợp chập 2 của 'abc' theo thứ tự từ điển: ['ab', 'ac', 'bc']
CombinationIterator  ['abc', 2]   -> None
next                 []           -> ab
hasNext              []           -> True
next                 []           -> ac
hasNext              []           -> True
next                 []           -> bc
hasNext              []           -> False


## 3. Kiến thức nền

### 3.1 Tổ hợp và số lượng `C(n, k)`

Tổ hợp chập `k` của `n` phần tử là một **tập con** `k` phần tử (không quan tâm thứ tự bên trong). Ta luôn viết các ký tự theo thứ tự xuất hiện trong `characters`. Số tổ hợp là `C(n, k) = n! / (k!(n−k)!)` — `math.comb(n, k)`. Lớn nhất ở bài này: `C(15, 7) = 6435`.

In [2]:
print("C(3,2) =", comb(3, 2), "| C(4,2) =", comb(4, 2), "| C(15,7) =", comb(15, 7))
print(max(comb(15, k) for k in range(1, 16)), "là số tổ hợp lớn nhất có thể")

C(3,2) = 3 | C(4,2) = 6 | C(15,7) = 6435
6435 là số tổ hợp lớn nhất có thể


### 3.2 Thứ tự từ điển

So sánh chuỗi như tra từ điển: so ký tự đầu, bằng nhau thì so ký tự tiếp. Python so sánh chuỗi đúng theo cách này. `itertools.combinations` trả tổ hợp theo thứ tự từ điển **của vị trí** — vì `characters` đã sắp xếp, đó cũng là thứ tự từ điển của chuỗi.

In [3]:
words = ["ac", "ab", "bc", "ad", "bd", "cd"]
print(sorted(words))
combs = ["".join(c) for c in combinations("abcd", 2)]
print(combs, "| đã đúng thứ tự từ điển?", combs == sorted(combs))

['ab', 'ac', 'ad', 'bc', 'bd', 'cd']
['ab', 'ac', 'ad', 'bc', 'bd', 'cd'] | đã đúng thứ tự từ điển? True


### 3.3 Lớp (class), thuộc tính và phương thức

`__init__` lưu trạng thái vào `self.xxx`; các phương thức khác đọc/sửa trạng thái đó. Mỗi đối tượng có trạng thái riêng.

In [4]:
class Counter3:
    def __init__(self, limit):
        self.i = 0
        self.limit = limit
    def next(self):
        self.i += 1
        return self.i
    def hasNext(self):
        return self.i < self.limit

c = Counter3(2)
print(c.next(), c.hasNext(), c.next(), c.hasNext())

1 True 2 False


### 3.4 Bộ chạy chuỗi lệnh kiểu LeetCode

LeetCode test bài thiết kế bằng hai mảng: tên lệnh và đối số. Lệnh đầu là tên lớp (gọi constructor, output `null` ↔ `None`). Ta viết `run_ops(cls, ops, args)` dùng `getattr` để gọi phương thức theo tên.

In [5]:
def run_ops(cls, ops, args):
    out, obj = [], None
    for op, a in zip(ops, args):
        if op == cls.__name__ or op == "CombinationIterator":
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print(run_ops(Counter3, ["Counter3", "next", "hasNext", "next", "hasNext"], [[2], [], [], [], []]))

[None, 1, True, 2, False]


### 3.5 Bitmask

Một số nguyên `n` bit biểu diễn một tập con: bit bật ⇒ chọn phần tử. `bin(x).count("1")` (hoặc `x.bit_count()` từ Python 3.10) đếm số bit 1. `x >> i & 1` kiểm tra bit `i`.

In [6]:
x = 0b1011
print(bin(x), "| số bit 1:", x.bit_count(), "| bit 2 bật?", x >> 2 & 1)
s = "abcd"
print("mask 1011 (bit 3 ↔ 'a') ->", "".join(s[i] for i in range(4) if x >> (3 - i) & 1))

0b1011 | số bit 1: 3 | bit 2 bật? 0
mask 1011 (bit 3 ↔ 'a') -> acd


## 4. Ý tưởng nền tảng / trực giác

Liệt kê tay các tổ hợp chập 2 của `"abcd"` theo thứ tự từ điển, kèm **chỉ số** các ký tự:

| # | tổ hợp | chỉ số |
|---|---|---|
| 0 | ab | (0, 1) |
| 1 | ac | (0, 2) |
| 2 | ad | (0, 3) |
| 3 | bc | (1, 2) |
| 4 | bd | (1, 3) |
| 5 | cd | (2, 3) |

Quan sát cách đi từ một tổ hợp sang tổ hợp kế:
- `(0,1) → (0,2) → (0,3)`: tăng **vị trí cuối**.
- `(0,3) → (1,2)`: vị trí cuối đã chạm trần (3 = n−1), nên tăng vị trí **trước đó** (0 → 1) rồi đặt lại vị trí sau ngay sát: 2.
- Tổ hợp cuối `(2,3)` = `(n−k, …, n−1)`: mọi vị trí đều chạm trần ⇒ hết.

Vị trí `i` (đếm từ 0) có **trần** là `n − k + i`.

In [7]:
n, k = 4, 2
for idx in combinations(range(n), k):
    caps = [n - k + i for i in range(k)]
    print(idx, "".join("abcd"[j] for j in idx), "| trần từng vị trí:", caps)

(0, 1) ab | trần từng vị trí: [2, 3]
(0, 2) ac | trần từng vị trí: [2, 3]
(0, 3) ad | trần từng vị trí: [2, 3]
(1, 2) bc | trần từng vị trí: [2, 3]
(1, 3) bd | trần từng vị trí: [2, 3]
(2, 3) cd | trần từng vị trí: [2, 3]


## 5. Cách 1 — Brute Force / baseline: sinh trước toàn bộ bằng `itertools`

**Ý tưởng:** ở `__init__`, sinh **tất cả** tổ hợp vào một list; giữ con trỏ `pos`. `next` trả `combos[pos]` rồi tăng `pos`; `hasNext` kiểm tra `pos < len(combos)`.

In [8]:
class CombinationIteratorPre:
    def __init__(self, characters: str, combinationLength: int):
        self.combos = ["".join(c) for c in combinations(characters, combinationLength)]
        self.pos = 0

    def next(self) -> str:
        s = self.combos[self.pos]
        self.pos += 1
        return s

    def hasNext(self) -> bool:
        return self.pos < len(self.combos)

out = run_ops(CombinationIteratorPre, EX_OPS, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, 'ab', True, 'ac', True, 'bc', False] | khớp: True


**Độ phức tạp:** `__init__` tốn `O(C(n,k)·k)` thời gian và bộ nhớ; `next`, `hasNext` đều `O(1)`. Với `n ≤ 15` thì tối đa `6435` chuỗi — hoàn toàn chấp nhận được.

**Nút thắt:** tốn bộ nhớ cho cả những tổ hợp **không bao giờ được hỏi tới** (ví dụ chỉ gọi `next` 3 lần). Nếu `n` lớn hơn (ví dụ 30, `C(30,15) ≈ 1.5·10^8`), cách này không khả thi. Ngoài ra, nhiều người phỏng vấn muốn bạn **tự sinh** tổ hợp thay vì gọi thư viện.

In [9]:
for n_ in [15, 20, 24]:
    k_ = n_ // 2
    print(f"n={n_}, k={k_}: C = {comb(n_, k_):>10,} tổ hợp -> ~{comb(n_, k_) * (k_ + 49) / 1e6:,.0f} MB nếu sinh trước")

n=15, k=7: C =      6,435 tổ hợp -> ~0 MB nếu sinh trước
n=20, k=10: C =    184,756 tổ hợp -> ~11 MB nếu sinh trước
n=24, k=12: C =  2,704,156 tổ hợp -> ~165 MB nếu sinh trước


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** thay vì lưu mọi tổ hợp, chỉ lưu **tổ hợp hiện tại** dưới dạng mảng chỉ số tăng dần `idx[0] < idx[1] < … < idx[k−1]`, rồi tính tổ hợp kế tiếp khi cần.

**Thuật toán tổ hợp kế tiếp:**
1. Tìm `i` **lớn nhất** sao cho `idx[i] < n − k + i` (chưa chạm trần).
2. Nếu không có `i` ⇒ đây là tổ hợp cuối.
3. Ngược lại: `idx[i] += 1`, rồi với `j > i`: `idx[j] = idx[j−1] + 1` (đặt đuôi sát nhất có thể ⇒ nhỏ nhất theo từ điển).

**Quy tắc "nếu… thì…":**
- Nếu vị trí cuối chưa chạm trần ⇒ chỉ tăng vị trí cuối.
- Nếu một vị trí tăng ⇒ toàn bộ phần đuôi sau nó **phải đặt lại** về giá trị nhỏ nhất.
- Nếu `idx[0] = n − k` ⇒ (mọi vị trí đều chạm trần) ⇒ hết tổ hợp.

**Thiết kế iterator:** giữ `self.cur` = tổ hợp sẽ trả ở lần `next` tới (hoặc `None` nếu hết). `next` trả `cur` và tính trước tổ hợp kế; `hasNext` chỉ cần `cur is not None` — `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Sinh trước bằng backtracking tự viết

**Từng đoạn:**
1. `bt(start, path)`: nếu `len(path) == k` thì lưu; ngược lại thử mọi ký tự `i ≥ start`.
2. Vì thử `i` tăng dần và chỉ chọn về bên phải, kết quả tự động theo thứ tự từ điển.
3. Cắt tỉa: chỉ thử `i` đến `n − (k − len(path))` để còn đủ ký tự.

In [10]:
class CombinationIteratorBT:
    def __init__(self, characters: str, combinationLength: int):
        n, k = len(characters), combinationLength
        self.combos = []
        path = []

        def bt(start):
            if len(path) == k:
                self.combos.append("".join(path))
                return
            for i in range(start, n - (k - len(path)) + 1):   # còn đủ ký tự
                path.append(characters[i])
                bt(i + 1)
                path.pop()

        bt(0)
        self.combos.reverse()          # để next() dùng pop() O(1) từ cuối

    def next(self) -> str:
        return self.combos.pop()

    def hasNext(self) -> bool:
        return bool(self.combos)

out = run_ops(CombinationIteratorBT, EX_OPS, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, 'ab', True, 'ac', True, 'bc', False] | khớp: True


**Độ phức tạp Cách 2:** `__init__` `O(C(n,k)·k)`, `next`/`hasNext` `O(1)`, bộ nhớ `O(C(n,k)·k)`. Mẹo đảo ngược list rồi `pop()` giúp không cần con trỏ.

### Cách 3 — Bitmask (lazy, chỉ lưu một số nguyên)

**Ý tưởng:** gán ký tự `characters[i]` cho bit `n−1−i` (ký tự đầu ↔ bit cao nhất). Khi đó, giữa các mask có đúng `k` bit 1, **mask lớn hơn ⇔ tổ hợp nhỏ hơn theo từ điển** (vì chọn ký tự sớm hơn ⇔ bật bit cao hơn). Vậy duyệt mask **giảm dần** từ `2^n − 1`, bỏ qua mask không đủ `k` bit.

**Từng đoạn:**
1. `self.mask` = mask ứng với tổ hợp kế tiếp (tìm trước).
2. `_advance(m)`: giảm `m` cho tới khi gặp mask có `k` bit, hoặc `m < 0`.
3. `next`: đổi mask thành chuỗi, rồi `_advance(mask − 1)`.

In [11]:
class CombinationIteratorMask:
    def __init__(self, characters: str, combinationLength: int):
        self.s = characters
        self.n = len(characters)
        self.k = combinationLength
        self.mask = self._advance((1 << self.n) - 1)

    def _advance(self, m):
        while m >= 0 and m.bit_count() != self.k:
            m -= 1
        return m                      # -1 nghĩa là hết

    def next(self) -> str:
        m, n = self.mask, self.n
        res = "".join(self.s[i] for i in range(n) if m >> (n - 1 - i) & 1)
        self.mask = self._advance(m - 1)
        return res

    def hasNext(self) -> bool:
        return self.mask >= 0

out = run_ops(CombinationIteratorMask, EX_OPS, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, 'ab', True, 'ac', True, 'bc', False] | khớp: True


**Độ phức tạp Cách 3:** bộ nhớ `O(1)` (ngoài chuỗi). Tổng thời gian duyệt hết mọi mask là `O(2^n)` ≈ 32768 bước khi `n = 15`, chia đều cho các lần `next` (worst một lần `next` có thể phải bỏ qua nhiều mask). Hoàn toàn đủ nhanh với `n ≤ 15`.

### Cách 4 — Sinh tổ hợp kế tiếp tại chỗ trên mảng chỉ số (lazy, `O(k)`)

Cài đặt thuật toán ở mục 6.

In [12]:
class CombinationIteratorLazy:
    def __init__(self, characters: str, combinationLength: int):
        self.s = characters
        self.n = len(characters)
        self.k = combinationLength
        self.idx = list(range(self.k))   # tổ hợp đầu tiên: 0, 1, ..., k-1
        self.has = True

    def next(self) -> str:
        res = "".join(self.s[i] for i in self.idx)
        n, k, idx = self.n, self.k, self.idx
        i = k - 1
        while i >= 0 and idx[i] == n - k + i:   # bỏ qua các vị trí đã chạm trần
            i -= 1
        if i < 0:
            self.has = False                     # vừa trả tổ hợp cuối cùng
        else:
            idx[i] += 1
            for j in range(i + 1, k):            # đặt lại phần đuôi
                idx[j] = idx[j - 1] + 1
        return res

    def hasNext(self) -> bool:
        return self.has

out = run_ops(CombinationIteratorLazy, EX_OPS, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, 'ab', True, 'ac', True, 'bc', False] | khớp: True


**Độ phức tạp Cách 4:** mỗi `next` `O(k)` (tạo chuỗi `k` ký tự + cập nhật chỉ số), `hasNext` `O(1)`, bộ nhớ `O(k)`. Đây là cách "chuẩn" khi người phỏng vấn yêu cầu không sinh trước.

### Cách 5 — Generator + nhìn trước một phần tử

Python có sẵn **generator** (`yield`). Khó khăn: generator không có `hasNext`. Giải pháp: luôn **nhìn trước** (prefetch) một phần tử và lưu vào `self.buf`.

In [13]:
class CombinationIteratorGen:
    _END = object()                       # giá trị canh "hết"

    def __init__(self, characters: str, combinationLength: int):
        self.gen = ("".join(c) for c in combinations(characters, combinationLength))
        self.buf = next(self.gen, self._END)

    def next(self) -> str:
        res = self.buf
        self.buf = next(self.gen, self._END)   # nhìn trước phần tử kế
        return res

    def hasNext(self) -> bool:
        return self.buf is not self._END

out = run_ops(CombinationIteratorGen, EX_OPS, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, 'ab', True, 'ac', True, 'bc', False] | khớp: True


**Độ phức tạp Cách 5:** mỗi `next` `O(k)` trung bình, bộ nhớ `O(k)`. Lưu ý trong lớp này, hàm dựng sẵn `next(...)` vẫn gọi được vì bên trong phương thức, tên `next` (không có `self.`) trỏ tới hàm built-in.

## 8. TRACE từng bước

Trace Cách 4 với `characters = "abcd"`, `k = 2` (trần: vị trí 0 → 2, vị trí 1 → 3):

| lần next | idx trước | trả về | i tìm được | idx sau |
|---|---|---|---|---|
| 1 | [0, 1] | ab | 1 | [0, 2] |
| 2 | [0, 2] | ac | 1 | [0, 3] |
| 3 | [0, 3] | ad | 0 (vị trí 1 chạm trần) | [1, 2] |
| 4 | [1, 2] | bc | 1 | [1, 3] |
| 5 | [1, 3] | bd | 0 | [2, 3] |
| 6 | [2, 3] | cd | không có ⇒ hết | — |

In [14]:
def trace_lazy(characters, k):
    n = len(characters)
    idx = list(range(k))
    step = 0
    while True:
        step += 1
        res = "".join(characters[i] for i in idx)
        before = idx[:]
        i = k - 1
        while i >= 0 and idx[i] == n - k + i:
            i -= 1
        if i < 0:
            print(f"next #{step}: idx={before} -> trả '{res}', không còn vị trí tăng được -> HẾT")
            break
        idx[i] += 1
        for j in range(i + 1, k):
            idx[j] = idx[j - 1] + 1
        print(f"next #{step}: idx={before} -> trả '{res}', tăng vị trí i={i}, idx mới={idx}")

trace_lazy("abcd", 2)

next #1: idx=[0, 1] -> trả 'ab', tăng vị trí i=1, idx mới=[0, 2]
next #2: idx=[0, 2] -> trả 'ac', tăng vị trí i=1, idx mới=[0, 3]
next #3: idx=[0, 3] -> trả 'ad', tăng vị trí i=0, idx mới=[1, 2]
next #4: idx=[1, 2] -> trả 'bc', tăng vị trí i=1, idx mới=[1, 3]
next #5: idx=[1, 3] -> trả 'bd', tăng vị trí i=0, idx mới=[2, 3]
next #6: idx=[2, 3] -> trả 'cd', không còn vị trí tăng được -> HẾT


In [15]:
def trace_mask(characters, k):
    n = len(characters)
    m = (1 << n) - 1
    while m >= 0:
        if m.bit_count() == k:
            s = "".join(characters[i] for i in range(n) if m >> (n - 1 - i) & 1)
            print(f"mask={m:0{n}b} ({m:>2}) -> '{s}'")
        m -= 1

trace_mask("abcd", 2)

mask=1100 (12) -> 'ab'
mask=1010 (10) -> 'ac'
mask=1001 ( 9) -> 'ad'
mask=0110 ( 6) -> 'bc'
mask=0101 ( 5) -> 'bd'
mask=0011 ( 3) -> 'cd'


## 9. Code nộp LeetCode

Chọn Cách 4 (lazy, tổ hợp kế tiếp tại chỗ): bộ nhớ `O(k)`, mỗi thao tác `O(k)`, không phụ thuộc thư viện. Đúng tên lớp và phương thức `CombinationIterator`, `next`, `hasNext`.

In [16]:
class CombinationIterator:

    def __init__(self, characters: str, combinationLength: int):
        self.chars = characters
        self.n = len(characters)
        self.k = combinationLength
        self.idx = list(range(combinationLength))   # tổ hợp nhỏ nhất
        self.has_next = True

    def next(self) -> str:
        n, k, idx = self.n, self.k, self.idx
        res = "".join(self.chars[i] for i in idx)
        i = k - 1
        while i >= 0 and idx[i] == n - k + i:      # vị trí i đã chạm trần
            i -= 1
        if i < 0:
            self.has_next = False
        else:
            idx[i] += 1
            for j in range(i + 1, k):
                idx[j] = idx[j - 1] + 1
        return res

    def hasNext(self) -> bool:
        return self.has_next


out = run_ops(CombinationIterator, EX_OPS, EX_ARGS)
print(out, "OK" if out == EX_OUT else "SAI")

[None, 'ab', True, 'ac', True, 'bc', False] OK


## 10. Vì sao thuật toán đúng

Xét các tổ hợp như dãy chỉ số tăng `idx`, so sánh theo thứ tự từ điển (trùng với thứ tự từ điển của chuỗi vì `characters` đã sắp xếp và ký tự phân biệt).

1. **Trần đúng:** vị trí `i` cần chừa chỗ cho `k − 1 − i` chỉ số lớn hơn phía sau, nên `idx[i] ≤ n − k + i`.
2. **Tổ hợp kế tiếp là nhỏ nhất trong các tổ hợp lớn hơn:** gọi `i` là vị trí phải nhất chưa chạm trần. Mọi vị trí sau `i` đã chạm trần ⇒ không thể lớn hơn nếu giữ nguyên tiền tố `idx[0..i]`. Vì vậy tổ hợp kế phải đổi vị trí `i` (hoặc sớm hơn); tăng `idx[i]` **ít nhất** (thêm 1) và đặt đuôi **nhỏ nhất** (`idx[i]+1, idx[i]+2, …`) cho tổ hợp nhỏ nhất có thể vượt tổ hợp hiện tại.
3. **Đuôi hợp lệ:** sau khi tăng, `idx[i] ≤ n − k + i` nên `idx[j] = idx[i] + (j − i) ≤ n − k + j` — mọi vị trí vẫn trong trần.
4. **Kết thúc đúng:** không tìm được `i` ⇔ `idx = (n−k, …, n−1)` — tổ hợp lớn nhất. Khi đó `has_next = False` ngay sau khi trả nó.
5. Bắt đầu từ `(0, …, k−1)` (nhỏ nhất) và mỗi bước sang đúng tổ hợp liền sau ⇒ liệt kê đủ `C(n,k)` tổ hợp theo đúng thứ tự.

## 11. Phân tích Time / Space

Gọi `n = len(characters)`, `k = combinationLength`, `N = C(n, k)`, `q` = số lời gọi.

| Cách | `__init__` | `next` | `hasNext` | Bộ nhớ |
|---|---|---|---|---|
| 1. itertools sinh trước | `O(N·k)` | `O(1)` | `O(1)` | `O(N·k)` |
| 2. Backtracking sinh trước | `O(N·k)` | `O(1)` | `O(1)` | `O(N·k)` |
| 3. Bitmask lazy | `O(2^(n−k))` bước để tìm mask đầu `(2^k−1)·2^(n−k)` | khấu hao `O(2^n / N + n)` | `O(1)` | `O(1)` |
| 4. Tổ hợp kế tiếp | `O(k)` | `O(k)` | `O(1)` | `O(k)` |
| 5. Generator + nhìn trước | `O(k)` | `O(k)` khấu hao | `O(1)` | `O(k)` |

Với `n ≤ 15`, `q ≤ 10^4`, mọi cách đều nhanh; khác biệt chủ yếu nằm ở **bộ nhớ** và **khả năng mở rộng**.

## 12. Bộ test

1. Ví dụ đề qua `run_ops`.
2. Duyệt **hết** mọi tổ hợp cho mọi `(n, k)` với `n ≤ 10`, so với `itertools.combinations`, và kiểm tra `hasNext` sau mỗi bước.
3. Chuỗi lệnh ngẫu nhiên (xen kẽ `next`/`hasNext`, chỉ gọi `next` khi còn — đúng ràng buộc) với seed cố định; so output của mọi cách.
4. Test hiệu năng `n = 15`, `k = 7`, gọi tới `10^4` lệnh.

In [17]:
IMPLS = [CombinationIteratorPre, CombinationIteratorBT, CombinationIteratorMask,
         CombinationIteratorLazy, CombinationIteratorGen, CombinationIterator]

for cls in IMPLS:
    assert run_ops(cls, EX_OPS, EX_ARGS) == EX_OUT, cls.__name__

letters = "abcdefghijklmno"
for n in range(1, 11):
    chars = letters[:n]
    for k in range(1, n + 1):
        want = ["".join(c) for c in combinations(chars, k)]
        for cls in IMPLS:
            it = cls(chars, k)
            got = []
            while it.hasNext():
                got.append(it.next())
                assert it.hasNext() == (len(got) < len(want)), (cls.__name__, chars, k)
            assert got == want, (cls.__name__, chars, k)
print("Duyệt hết mọi (n, k) với n <= 10: cả", len(IMPLS), "cách đúng thứ tự và đúng hasNext")

Duyệt hết mọi (n, k) với n <= 10: cả 6 cách đúng thứ tự và đúng hasNext


In [18]:
def random_ops(rng, max_calls=60):
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    n = rng.randint(1, 15)
    chars = "".join(sorted(rng.sample(alphabet, n)))     # phân biệt, đã sắp xếp
    k = rng.randint(1, n)
    total = comb(n, k)
    ops, args, used = ["CombinationIterator"], [[chars, k]], 0
    for _ in range(rng.randint(1, max_calls)):
        if used < total and rng.random() < 0.6:
            ops.append("next"); args.append([]); used += 1
        else:
            ops.append("hasNext"); args.append([])
    return ops, args

rng = random.Random(1286)
for _ in range(400):
    ops, args = random_ops(rng)
    ref = run_ops(CombinationIteratorPre, ops, args)
    for cls in IMPLS[1:]:
        assert run_ops(cls, ops, args) == ref, (cls.__name__, ops, args)
print("400 chuỗi lệnh ngẫu nhiên: mọi cách cho cùng output")

400 chuỗi lệnh ngẫu nhiên: mọi cách cho cùng output


In [19]:
chars = letters                      # n = 15
for cls in IMPLS:
    t = time.perf_counter()
    it = cls(chars, 7)
    calls = 0
    while calls < 10**4 and it.hasNext():
        it.next()
        calls += 2
    print(f"{cls.__name__:<26} 10^4 lời gọi (n=15, k=7): {time.perf_counter() - t:.3f}s")

CombinationIteratorPre     10^4 lời gọi (n=15, k=7): 0.002s
CombinationIteratorBT      10^4 lời gọi (n=15, k=7): 0.002s
CombinationIteratorMask    10^4 lời gọi (n=15, k=7): 0.008s
CombinationIteratorLazy    10^4 lời gọi (n=15, k=7): 0.005s
CombinationIteratorGen     10^4 lời gọi (n=15, k=7): 0.001s
CombinationIterator        10^4 lời gọi (n=15, k=7): 0.004s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `hasNext` lệch 1: `pos < len(combos) - 1`

In [20]:
class WrongHasNext(CombinationIteratorPre):
    def hasNext(self) -> bool:
        return self.pos < len(self.combos) - 1     # SAI

ops = ["CombinationIterator", "next", "hasNext", "next", "hasNext"]
args = [["abc", 2], [], [], [], []]
print("Code sai :", run_ops(WrongHasNext, ops, args))
print("Code đúng:", run_ops(CombinationIterator, ops, args))

Code sai : [None, 'ab', True, 'ac', False]
Code đúng: [None, 'ab', True, 'ac', True]


Sau khi đã trả `"ab"` và `"ac"`, vẫn còn `"bc"` nên `hasNext` phải là `True`. **Code đúng:** `pos < len(combos)` — "con trỏ còn nằm trong mảng".

### Lỗi 2 — Tăng một vị trí nhưng quên đặt lại phần đuôi

In [21]:
class WrongNoReset(CombinationIteratorLazy):
    def next(self) -> str:
        res = "".join(self.s[i] for i in self.idx)
        n, k, idx = self.n, self.k, self.idx
        i = k - 1
        while i >= 0 and idx[i] == n - k + i:
            i -= 1
        if i < 0:
            self.has = False
        else:
            idx[i] += 1                              # SAI: không đặt lại idx[i+1:]
        return res

def drain(cls, chars, k):
    it, out = cls(chars, k), []
    while it.hasNext() and len(out) < 20:
        out.append(it.next())
    return out

print("Code sai :", drain(WrongNoReset, "abcd", 2))
print("Code đúng:", drain(CombinationIterator, "abcd", 2))

Code sai : ['ab', 'ac', 'ad', 'bd', 'cd']
Code đúng: ['ab', 'ac', 'ad', 'bc', 'bd', 'cd']


Sau `"ad"` (idx `[0,3]`), code sai tăng `idx[0]` thành 1 nhưng giữ `idx[1] = 3` ⇒ `"bd"`, bỏ sót `"bc"`. **Code đúng:** sau khi tăng `idx[i]`, gán `idx[j] = idx[j−1] + 1` cho `j > i`.

### Lỗi 3 — Bitmask duyệt **tăng dần** với bit `i` ↔ ký tự `i`

In [22]:
def wrong_mask_order(chars, k):
    n = len(chars)
    res = []
    for m in range(1 << n):                          # SAI về thứ tự
        if m.bit_count() == k:
            res.append("".join(chars[i] for i in range(n) if m >> i & 1))
    return res

print("Code sai :", wrong_mask_order("abcd", 2))
print("Code đúng:", drain(CombinationIteratorMask, "abcd", 2))

Code sai : ['ab', 'ac', 'bc', 'ad', 'bd', 'cd']
Code đúng: ['ab', 'ac', 'ad', 'bc', 'bd', 'cd']


Thứ tự số nguyên tăng dần **không** trùng thứ tự từ điển: `"ad"` (mask 9) đứng sau `"bc"` (mask 6). **Code đúng:** ký tự đầu ↔ bit cao nhất và duyệt mask **giảm dần** (Cách 3).

### Lỗi 4 — Dùng generator nhưng `hasNext` "ăn mất" phần tử

In [23]:
class WrongGen:
    def __init__(self, characters, combinationLength):
        self.gen = ("".join(c) for c in combinations(characters, combinationLength))
    def next(self):
        return next(self.gen)
    def hasNext(self):
        try:
            next(self.gen)                           # SAI: lấy mất một tổ hợp
            return True
        except StopIteration:
            return False

ops4 = ["CombinationIterator", "next", "hasNext", "next"]
args4 = [["abcd", 2], [], [], []]
print("Code sai :", run_ops(WrongGen, ops4, args4), "<- next thứ hai lẽ ra là 'ac'")
print("Code đúng:", run_ops(CombinationIteratorGen, ops4, args4))

Code sai : [None, 'ab', True, 'ad'] <- next thứ hai lẽ ra là 'ac'
Code đúng: [None, 'ab', True, 'ac']


**Code đúng:** **nhìn trước** một phần tử và lưu trong `buf` (Cách 5); `hasNext` chỉ đọc `buf`, không tiêu thụ generator.

### Lỗi 5 — Biến trạng thái đặt ở mức lớp (dùng chung giữa các đối tượng)

In [24]:
class WrongShared:
    idx = [0, 1]                       # SAI: thuộc tính lớp, mọi đối tượng dùng chung
    def __init__(self, characters, combinationLength):
        self.s = characters
    def next(self):
        res = "".join(self.s[i] for i in self.idx)
        self.idx[1] += 1               # sửa list dùng chung
        return res

a, b = WrongShared("abcd", 2), WrongShared("abcd", 2)
print("Code sai : a.next() =", a.next(), "| b.next() =", b.next(), "(lẽ ra b cũng trả 'ab')")
a, b = CombinationIterator("abcd", 2), CombinationIterator("abcd", 2)
print("Code đúng: a.next() =", a.next(), "| b.next() =", b.next())

Code sai : a.next() = ab | b.next() = ac (lẽ ra b cũng trả 'ab')
Code đúng: a.next() = ab | b.next() = ab


**Code đúng:** khởi tạo mọi trạng thái trong `__init__` qua `self.xxx = ...`. LeetCode tạo nhiều đối tượng trong cùng một lần chạy test, nên lỗi này rất hay gây "sai bí ẩn".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài **iterator**: luôn nghĩ "trạng thái tối thiểu là gì để sinh phần tử kế tiếp?" và "làm sao `hasNext` là `O(1)`?" ⇒ thường **tính trước/nhìn trước** một bước.
- `n ≤ 15` ⇒ sinh trước toàn bộ (`2^15`) hoàn toàn ổn — đừng sợ, nhưng nên biết cách lazy để trả lời câu hỏi mở rộng.
- Tổ hợp theo thứ tự từ điển ⇔ dãy chỉ số tăng; "kế tiếp" = tìm vị trí phải nhất còn tăng được (giống **next permutation**).
- Bitmask: muốn thứ tự từ điển thì phần tử đầu ↔ **bit cao**, duyệt **giảm dần**.
- Viết sẵn `run_ops` cho mọi bài Design để tự test nhanh.

## 15. Biến thể của bài

1. **LC 77 Combinations:** trả về mọi tổ hợp chập `k` của `1..n` — chính là Cách 2.
2. **LC 31 Next Permutation:** hoán vị kế tiếp — cùng tư duy "tìm vị trí phải nhất tăng được, đặt lại đuôi".
3. **LC 284 Peeking Iterator:** thêm `peek()` cho iterator — chính là kỹ thuật nhìn trước của Cách 5.
4. **LC 173 Binary Search Tree Iterator**, **LC 341 Flatten Nested List Iterator:** iterator lazy trên cấu trúc khác.
5. **Ký tự có thể lặp** (ví dụ `"aab"`): phải bỏ tổ hợp trùng (LC 40 Combination Sum II tư duy tương tự).
6. **Nhảy tới tổ hợp thứ `r`** (unrank) hoặc tính thứ hạng của một tổ hợp (rank) bằng `math.comb` — xem Bài tập 5.

In [25]:
def next_permutation(a):          # LC 31 — cùng khuôn "tìm vị trí phải nhất tăng được"
    i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]:
        i -= 1
    if i >= 0:
        j = len(a) - 1
        while a[j] <= a[i]:
            j -= 1
        a[i], a[j] = a[j], a[i]
    a[i + 1:] = reversed(a[i + 1:])
    return a

print(next_permutation([1, 2, 3]), next_permutation([1, 3, 2]), next_permutation([3, 2, 1]))

[1, 3, 2] [2, 1, 3] [1, 2, 3]


## 16. So sánh các cách

| Cách | Ý tưởng | `next` | Bộ nhớ | Khi nào dùng |
|---|---|---|---|---|
| 1. itertools sinh trước | list mọi tổ hợp + con trỏ | `O(1)` | `O(C(n,k)·k)` | nhanh gọn khi `n` nhỏ |
| 2. Backtracking sinh trước | tự sinh, đảo list, `pop()` | `O(1)` | `O(C(n,k)·k)` | khi không được dùng thư viện |
| 3. Bitmask lazy | mask giảm dần, đủ `k` bit | khấu hao nhỏ | `O(1)` | `n` nhỏ, thích bit |
| 4. Tổ hợp kế tiếp | tăng vị trí phải nhất, đặt lại đuôi | `O(k)` | `O(k)` | **nộp bài**, mở rộng tốt |
| 5. Generator + nhìn trước | `yield` + `buf` | `O(k)` | `O(k)` | code Pythonic, lazy |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** Với `["CombinationIterator","next","next","hasNext","next","next","hasNext"]` và `[["abcd",3],[],[],[],[],[],[]]`, output là gì?

**Bài 2 (tính tay):** Trong `CombinationIterator("abcde", 3)`, tổ hợp ngay sau `"acd"` là gì? Ngay sau `"ade"`? Tổ hợp thứ 7 (đếm từ 1) là gì?

**Bài 3 (điền TODO):** Hoàn thiện `next_combination_todo(idx, n)`: sửa `idx` tại chỗ thành tổ hợp kế tiếp và trả `True`, hoặc trả `False` nếu `idx` đã là tổ hợp cuối.

**Bài 4 (tự viết lại):** Tự viết lại lớp iterator dùng bitmask nhưng **sinh trước** danh sách mask hợp lệ (giảm dần) trong `__init__`.

**Bài 5 (mini challenge):** Viết `rank(chars, comb_str)` trả thứ hạng (từ 0) của một tổ hợp, và `unrank(chars, k, r)` trả tổ hợp có thứ hạng `r`, dùng `math.comb`, không liệt kê.

In [26]:
def next_combination_todo(idx, n):
    k = len(idx)
    i = k - 1
    # TODO 1: lùi i khi idx[i] đã chạm trần n - k + i
    # TODO 2: nếu i < 0 trả False
    # TODO 3: tăng idx[i], đặt lại idx[j] = idx[j-1] + 1 với j > i, trả True
    pass

class MaskListIteratorTodo:
    def __init__(self, characters, combinationLength):
        # TODO: tạo self.masks = các mask có đúng k bit, theo thứ tự giảm dần
        pass

# Lưu ý: cell này chỉ định nghĩa hàm/lớp, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Liệt kê `C(4,3) = 4` tổ hợp. (2) Theo thứ tự: `abc, abd, acd, bcd`. (3) Ghép với chuỗi lệnh, nhớ constructor trả `None`.

**Bài 2.** (1) `"acd"` = idx `[0,2,3]`, trần `[2,3,4]`. (2) Vị trí phải nhất chưa chạm trần là vị trí 2 (3 < 4) ⇒ `[0,2,4]`. `"ade"` = `[0,3,4]` ⇒ chỉ vị trí 0 tăng được ⇒ `[1,2,3]`. (3) Liệt kê: abc, abd, abe, acd, ace, ade, **bcd**.

**Bài 3.** (1) Vòng `while i >= 0 and idx[i] == n - k + i`. (2) Trả `False` nếu hết. (3) Đặt lại đuôi rồi trả `True`.

**Bài 4.** (1) `[m for m in range((1 << n) - 1, -1, -1) if m.bit_count() == k]`. (2) Con trỏ `pos`. (3) Đổi mask → chuỗi bằng bit `n−1−i`.

**Bài 5.** (1) Duyệt các vị trí của tổ hợp; với mỗi vị trí, đếm số tổ hợp có cùng tiền tố nhưng chọn ký tự nhỏ hơn ở vị trí đó: cộng `comb(n − 1 − c, còn_lại − 1)` cho mỗi ký tự `c` bị bỏ qua. (2) `unrank`: đi qua từng ký tự, nếu `r < comb(số ký tự sau nó, còn_lại − 1)` thì chọn nó, ngược lại trừ đi và bỏ qua. (3) Kiểm tra `unrank(rank(x)) == x` cho mọi tổ hợp nhỏ.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
ops1 = ["CombinationIterator", "next", "next", "hasNext", "next", "next", "hasNext"]
args1 = [["abcd", 3], [], [], [], [], [], []]
assert run_ops(CombinationIterator, ops1, args1) == [None, "abc", "abd", True, "acd", "bcd", False]

# Bài 2
all5 = ["".join(c) for c in combinations("abcde", 3)]
assert all5[all5.index("acd") + 1] == "ace"
assert all5[all5.index("ade") + 1] == "bcd"
assert all5[6] == "bcd"

# Bài 3
def next_combination_sol(idx, n):
    k = len(idx)
    i = k - 1
    while i >= 0 and idx[i] == n - k + i:
        i -= 1
    if i < 0:
        return False
    idx[i] += 1
    for j in range(i + 1, k):
        idx[j] = idx[j - 1] + 1
    return True

# Bài 4
class MaskListIteratorSol:
    def __init__(self, characters, combinationLength):
        n, k = len(characters), combinationLength
        self.s, self.n = characters, n
        self.masks = [m for m in range((1 << n) - 1, -1, -1) if m.bit_count() == k]
        self.pos = 0
    def next(self):
        m = self.masks[self.pos]
        self.pos += 1
        return "".join(self.s[i] for i in range(self.n) if m >> (self.n - 1 - i) & 1)
    def hasNext(self):
        return self.pos < len(self.masks)

# Bài 5
def rank_sol(chars, comb_str):
    n, k = len(chars), len(comb_str)
    pos = [chars.index(ch) for ch in comb_str]
    r, start = 0, 0
    for t, p in enumerate(pos):
        remaining = k - t
        for c in range(start, p):                 # các ký tự nhỏ hơn bị bỏ qua
            r += comb(n - 1 - c, remaining - 1)
        start = p + 1
    return r

def unrank_sol(chars, k, r):
    n, res = len(chars), []
    c = 0
    while len(res) < k:
        remaining = k - len(res)
        cnt = comb(n - 1 - c, remaining - 1)      # số tổ hợp nếu chọn chars[c]
        if r < cnt:
            res.append(chars[c])
        else:
            r -= cnt
        c += 1
    return "".join(res)

for n in range(1, 9):
    chars = letters[:n]
    for k in range(1, n + 1):
        want = ["".join(c) for c in combinations(chars, k)]
        idx, got = list(range(k)), []
        while True:
            got.append("".join(chars[i] for i in idx))
            if not next_combination_sol(idx, n):
                break
        assert got == want
        assert drain(MaskListIteratorSol, chars, k) == want[:20]
        for r, s in enumerate(want):
            assert rank_sol(chars, s) == r and unrank_sol(chars, k, r) == s
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tổ hợp kế tiếp (chỉ số tăng dần):**

```python
def next_combination(idx, n):
    k = len(idx); i = k - 1
    while i >= 0 and idx[i] == n - k + i:
        i -= 1
    if i < 0: return False
    idx[i] += 1
    for j in range(i + 1, k):
        idx[j] = idx[j - 1] + 1
    return True
```

**Template iterator nhìn trước:**

```python
self.buf = next(gen, END)
def next(self):    res = self.buf; self.buf = next(self.gen, END); return res
def hasNext(self): return self.buf is not END
```

**Checklist trước khi submit:**
- [ ] Tên lớp/phương thức đúng: `CombinationIterator`, `next`, `hasNext`.
- [ ] Trạng thái khởi tạo trong `__init__` (không dùng thuộc tính lớp dạng list).
- [ ] `hasNext` không làm thay đổi trạng thái.
- [ ] Tổ hợp đầu tiên `(0..k−1)`, tổ hợp cuối `(n−k..n−1)`; `hasNext` sai ngay sau khi trả tổ hợp cuối.
- [ ] Sau khi tăng một vị trí, **đặt lại đuôi**.
- [ ] Test bằng `run_ops` và bằng cách duyệt hết, so với `itertools.combinations`.